# TẠO MÔ HÌNH 3D (.GLB) VỚI HUNYUAN3D-2.1 TRÊN COLAB PRO
**Khuyến nghị GPU:** Chọn **L4 (24 GB VRAM)** hoặc **A100 (40 GB VRAM)** (`High-RAM`).
1. Chạy **Cell 1** để tải mã nguồn Hunyuan3D-2.1.
2. Chạy **Cell 2** để cài thư viện & biên dịch bộ tô màu PBR (~3–5 phút).
3. Chạy **Cell 3** để mở link giao diện web `trycloudflare.com` (nếu dùng A100 40GB có thể bỏ `--low_vram_mode`).

In [ ]:
# CELL 1: TẢI MÃ NGUỒN HUNYUAN3D-2.1
import os
if not os.path.exists("/content/Hunyuan3D-2.1"):
    !git clone https://github.com/Tencent-Hunyuan/Hunyuan3D-2.1.git /content/Hunyuan3D-2.1
%cd /content/Hunyuan3D-2.1

In [ ]:
# CELL 2: CÀI THƯ VIỆN & BIÊN DỊCH BỘ TÔ MÀU PBR (~3-5 PHÚT)
%cd /content/Hunyuan3D-2.1
!sed -i 's/^torch/# torch/' requirements.txt
!sed -i 's/^torchvision/# torchvision/' requirements.txt
!sed -i 's/^torchaudio/# torchaudio/' requirements.txt
!pip install -q -r requirements.txt

%cd /content/Hunyuan3D-2.1/hy3dpaint/custom_rasterizer
!pip install -q -e .

%cd /content/Hunyuan3D-2.1/hy3dpaint/DifferentiableRenderer
!bash compile_mesh_painter.sh

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

%cd /content/Hunyuan3D-2.1

In [ ]:
# CELL 3: KHỞI CHẠY GIAO DIỆN WEB TẠO MODEL 3D (.GLB)
import subprocess, time, re, torch

vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3) if torch.cuda.is_available() else 0
low_vram_flag = "" if vram_gb >= 35 else "--low_vram_mode"
print(f"GPU VRAM: {vram_gb:.1f} GB | Flag: {low_vram_flag or 'Full VRAM'}")

log_file = open("tunnel.log", "w")
subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:7860"],
    stdout=log_file,
    stderr=log_file,
)
time.sleep(5)

with open("tunnel.log") as f:
    urls = re.findall(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", f.read())
print(">>> Link giao diện 3D của bạn:", urls[0] if urls else "Kiểm tra file tunnel.log")

!python gradio_app.py --host 127.0.0.1 --port 7860 {low_vram_flag}